In [0]:
#Bibliotecas de inicialização
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from datetime import date, datetime, timedelta
from pyspark.sql.window import Window
GOLD_BD = "gold"

In [0]:

spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.{GOLD_BD}")

df_filmes = spark.table("workspace.silver.tb_info_filmes")
janela = Window.orderBy("id_filme")

#Cria gold.dim_movies
df_dim_movies = df_filmes.select(
    F.row_number().over(janela).try_cast("bigint").alias("sk_movie_id"),
    F.col("id_filme").try_cast("string"),
    F.col("titulo").try_cast("string"),
    F.col("data_lancamento").try_cast("date"),
    F.col("ano_lancamento").try_cast("int"),
    F.col("duracao_minutos").try_cast("int"),
    F.col("idioma_original").try_cast("string"),
    F.col("status_filme").try_cast("string"),
    F.col("sinopse").try_cast("string"),
)

display(df_dim_movies)

df_dim_movies.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"workspace.{GOLD_BD}.dim_movies")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:


df_financeiro = spark.table("workspace.silver.tb_financeiro_filmes")
df_metricas   = spark.table("workspace.silver.tb_metricas_engajamento")
df_dim_movies = spark.table(f"workspace.{GOLD_BD}.dim_movies")

df_fact_movie_performance = (
    df_dim_movies
    .filter(F.col("status_filme") == "Lançado")
    .select("sk_movie_id", "id_filme")
    .join(df_financeiro, on="id_filme", how="left")
    .join(df_metricas,   on="id_filme", how="left")
    .select(
        F.col("sk_movie_id"),
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int"),
    )
)

display(df_fact_movie_performance)

df_fact_movie_performance.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"workspace.{GOLD_BD}.fact_movies_performance")

sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1,null,null,0.00,null,null,0.00,1.132,0.0,0,6.8,27
2,null,null,0.00,null,null,0.00,0.6,0.0,0,null,40
3,null,null,0.00,null,null,0.00,0.6,0.0,0,4.7,10
4,null,null,0.00,null,null,0.00,1.169,0.0,0,4.8,16
5,null,null,0.00,null,null,0.00,0.6,0.0,0,7.2,15
6,null,null,0.00,null,null,0.00,0.615,0.0,0,null,25
7,null,null,0.00,null,null,0.00,0.6,0.0,0,5.9,10
8,4700000.00,null,-4700000.00,23885400.00,null,-23885400.00,1.489,6.75,6,6.2,382
9,null,null,0.00,null,null,0.00,0.6,0.0,0,7.7,25
10,6000000.00,null,-6000000.00,30492000.00,null,-30492000.00,13.212,6.8,15,null,236


In [0]:
df_generos = spark.table("workspace.silver.tb_generos")

df_dim_genres = (
    df_generos
    .select(F.col("genero").alias("nome_genero"))
    .filter(F.col("nome_genero").isNotNull())
    .distinct()
    .withColumn("sk_genre_id", F.monotonically_increasing_id())
    .select("sk_genre_id", "nome_genero")
)

display(df_dim_genres)

df_dim_genres.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.dim_genres")

sk_genre_id,nome_genero
0,Music
1,Family
2,Drama
3,Horror
4,History
5,Adventure
6,Comedy
7,Fantasy
8,Romance
9,Thriller


In [0]:
df_dim_people = (
    spark.table("workspace.silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa"),
    )
    .distinct()
    .withColumn("sk_person_id", F.monotonically_increasing_id())
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)

display(df_dim_people)

df_dim_people.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.dim_people")




sk_person_id,nome_pessoa,tipo_pessoa
0,Kurt Braunohler,Ator
1,José Antonio García,Ator
2,Shree Troietta Crooks,Ator
3,Miyu Tomita,Ator
4,Sephora Pondi,Ator
5,Michael Jackson,Ator
6,Anna Galvin,Ator
7,Mercedes Lambre,Ator
8,Gwendoline Hamon,Ator
9,Se-ah Jang,Ator


In [0]:
TAMANHO_MINIMO = 6   # nomes com menos de 5 caracteres são descartados

df_dim_companies = (
    spark.table("workspace.silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora"))
    .distinct()
    .filter(F.length("nome_produtora") >= TAMANHO_MINIMO) # São filtradas as produtoras com menos de 6 letras
    .withColumn("sk_company_id", F.monotonically_increasing_id())
    .select("sk_company_id", "nome_produtora")
)

display(df_dim_companies.orderBy(F.length("nome_produtora")).limit(50))

df_dim_companies.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.dim_companies")

sk_company_id,nome_produtora
3266,拍板影視文化
4039,Defilm
2512,Lantis
3164,Buemon
4297,Gmouse
3685,Murmur
2051,Format
2487,Filmax
2720,Filmin
3012,Viatel


In [0]:
print("Produtoras:", df_dim_companies.count(), "| Distintas:", df_dim_companies.select("nome_produtora").distinct().count())

# nomes curtos demais são onde o lixo costuma sobrar
display(df_dim_companies.orderBy(F.length("nome_produtora")).limit(50))

Produtoras: 41635 | Distintas: 41635


sk_company_id,nome_produtora
3266,拍板影視文化
4039,Defilm
2512,Lantis
3164,Buemon
4297,Gmouse
3685,Murmur
2051,Format
2487,Filmax
2720,Filmin
3012,Viatel


In [0]:
df_avaliacoes = spark.table("workspace.silver.tb_avaliacoes_usuarios")
df_dim_movies = spark.table("workspace.gold.dim_movies")

df_dim_reviews = (
    df_avaliacoes
    .groupBy("id_filme")
    .agg(
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios"),
    )
    .join(df_dim_movies.select("id_filme", "sk_movie_id"), on="id_filme", how="inner")
    .withColumn("sk_review_id", F.monotonically_increasing_id())
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
)

display(df_dim_reviews)

df_dim_reviews.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.dim_reviews")

sk_review_id,sk_movie_id,qtd_avaliacoes_usuarios,nota_media_usuarios
0,1,1,0.1
1,5,1,3.3
2,8,1,6.8
3,17,1,5.4
4,19,1,5.3
5,20,1,4.4
6,23,3,4.75
7,24,1,4.8
8,26,1,6.6
9,33,1,7.5


In [0]:
dim_movies = spark.table("workspace.gold.dim_movies").select("id_filme", "sk_movie_id")
dim_genres = spark.table("workspace.gold.dim_genres")

df_bridge_genre = (
    spark.table("workspace.silver.tb_generos")
    .join(dim_movies, on="id_filme", how="inner")
    .join(dim_genres, F.col("genero") == F.col("nome_genero"), how="inner")
    .select("sk_movie_id", "sk_genre_id")
    .distinct()
)

display(df_bridge_genre)

df_bridge_genre.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.bridge_movie_genre")

sk_movie_id,sk_genre_id
31843,4
20259,7
83702,10
62714,14
58868,5
29702,6
18921,7
28912,4
23793,7
65934,14


In [0]:
dim_people = spark.table("workspace.gold.dim_people")

df_bridge_person = (
    spark.table("workspace.silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .join(dim_movies, on="id_filme", how="inner")
    .join(
        dim_people,
        (F.col("nome_entidade") == F.col("nome_pessoa")) & (F.col("tipo_entidade") == F.col("tipo_pessoa")),
        how="inner",
    )
    .select("sk_movie_id", "sk_person_id")
    .distinct()
)

display(df_bridge_person)

df_bridge_person.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.bridge_movie_person")

sk_movie_id,sk_person_id
30893,93072
70452,138417
5922,244743
34775,143532
24049,79018
18535,225219
21532,196063
54265,29791
23681,187744
31638,139313


In [0]:
dim_companies = spark.table("workspace.gold.dim_companies")

df_bridge_company = (
    spark.table("workspace.silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .join(dim_movies, on="id_filme", how="inner")
    .join(dim_companies, F.col("nome_entidade") == F.col("nome_produtora"), how="inner")
    .select("sk_movie_id", "sk_company_id")
    .distinct()
)

display(df_bridge_company)

df_bridge_company.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.bridge_movie_company")

sk_movie_id,sk_company_id
8,13572
12,37970
88,16690
191,22476
274,25345
290,6496
291,6496
305,5934
334,31447
382,30462


In [0]:
#1. Qual é a receita total (em R$) somada de todos os filmes da base?
df_fato = spark.table("workspace.gold.fact_movies_performance")

# soma da receita em reais e quantidade de filmes com receita informada
df_receita_total = df_fato.agg(
    F.sum("receita_brl").alias("receita_total_brl"),
    F.count("receita_brl").alias("filmes_com_receita"),
)

# formata o total para leitura
df_receita_total = df_receita_total.withColumn(
    "receita_total_formatada", F.format_number("receita_total_brl", 2)
)

display(df_receita_total)

receita_total_brl,filmes_com_receita,receita_total_formatada
825603087431.54,3270,"825,603,087,431.54"


In [0]:
#2. Quais são os 5 filmes com maior popularidade? Mostre título e valor de popularidade

df_fato = spark.table("workspace.gold.fact_movies_performance")
df_filmes = spark.table("workspace.gold.dim_movies").select("sk_movie_id", "titulo")

# junta a popularidade com o título
df_popularidade = df_fato.join(df_filmes, on="sk_movie_id").select("titulo", "popularidade")

# ordena do mais popular para o menos e pega os 5 primeiros
df_top5_popularidade = df_popularidade.orderBy(F.desc("popularidade")).limit(5)

display(df_top5_popularidade)

titulo,popularidade
Nightwish: Decades (Live in Buenos Aires),2.018123410567891E38
Love: Augmented,1.0252800328164457E21
Silent Displacement – The Unknown Deportation of Ingria,6.300019465E12
Asylum of the Devil,2.0093201020121E12
Sandy & Junior: Nossa História,2.0192007E7


In [0]:
#3. Quantos filmes cada gênero possui? Liste do maior para o menor volume.

df_bridge_genero = spark.table("workspace.gold.bridge_movie_genre")
df_generos = spark.table("workspace.gold.dim_genres")

# traz o nome do gênero para cada par filme-gênero
df_filme_genero = df_bridge_genero.join(df_generos, on="sk_genre_id")

# conta os filmes distintos de cada gênero
df_filmes_por_genero = (
    df_filme_genero
    .groupBy("nome_genero")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_filmes"))
)

df_filmes_por_genero = df_filmes_por_genero.orderBy(F.desc("qtd_filmes"))
display(df_filmes_por_genero)


nome_genero,qtd_filmes
Drama,32291
Documentary,19003
Comedy,18628
Thriller,10279
Horror,9732
Romance,7640
Action,6049
Crime,4747
Animation,4473
TV Movie,4080


In [0]:
#4. Para os 10 filmes de maior receita, mostre título, receita (em US$ e R$) e a posição de cada um no ranking

df_fato   = spark.table("workspace.gold.fact_movies_performance")
df_filmes = spark.table("workspace.gold.dim_movies").select("sk_movie_id", "titulo")

# só filmes com receita informada, já com o título
df_receita = (
    df_fato
    .join(df_filmes, on="sk_movie_id")
    .select("titulo", "receita_usd", "receita_brl")
)

# ranking pela receita em dólar, da maior para a menor
janela_receita = Window.orderBy(F.desc("receita_usd"))
df_ranking = df_receita.withColumn("posicao", F.rank().over(janela_receita))

# mantém o top 10 e organiza as colunas
df_top10_receita = (
    df_ranking
    .filter(F.col("posicao") <= 10)
    .select("posicao", "titulo", "receita_usd", "receita_brl")
    .orderBy("posicao")
)

display(df_top10_receita)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


posicao,titulo,receita_usd,receita_brl
1,Avengers: Endgame,2800000000.00,14229600000.00
2,Avatar: The Way of Water,2320250281.00,11791511928.04
3,AVENGERS: INFINITY WAR,2052415039.00,10430373228.20
4,spider-man: no way home,1921847111.00,9766827018.10
5,The Lion King,1663075401.00,8451749187.88
6,Top Gun: Maverick,1488732821.00,7565740196.32
7,Barbie,1428545028.00,7259865832.30
8,The Super Mario Bros. Movie,1355725263.00,6889795786.57
9,Black Panther,1349926083.00,6860324353.81
10,Star Wars: The Last Jedi,1332698830.00,6772775454.06


In [0]:
#5. Qual ator teve a maior quantidade de participações nos filmes lançados nos últimos 2 anos?

ANOS=2

df_filmes  = spark.table("workspace.gold.dim_movies")
df_bridge  = spark.table("workspace.gold.bridge_movie_person")
df_pessoas = spark.table("workspace.gold.dim_people")

data_fim = date.today()
data_inicio = data_fim.replace(year=data_fim.year - ANOS)

df_filmes_periodo = (
    df_filmes
    .filter(F.col("status_filme") == "Lançado")
    .filter(F.col("data_lancamento").between(F.lit(data_inicio), F.lit(data_fim)))
    .select("sk_movie_id")
)
print("Filmes no período:", df_filmes_periodo.count())

# atores desses filmes
df_atores_periodo = (
    df_filmes_periodo
    .join(df_bridge, on="sk_movie_id")
    .join(df_pessoas.filter(F.col("tipo_pessoa") == "Ator"), on="sk_person_id")
)

# participações por ator
df_participacoes = (
    df_atores_periodo
    .groupBy("nome_pessoa")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_participacoes"))
)

df_ranking_atores = (
    df_participacoes
    .orderBy(F.desc("qtd_participacoes"), "nome_pessoa")
    .limit(1)
)

display(df_ranking_atores)

Filmes no período: 54


nome_pessoa,qtd_participacoes
Austin Rearden,4


In [0]:
#6. Qual a produtora de filmes teve o maior Lucro nos últimos 5 anos?

ANOS=5

df_filmes      = spark.table("workspace.gold.dim_movies")
df_fato        = spark.table("workspace.gold.fact_movies_performance")
df_bridge      = spark.table("workspace.gold.bridge_movie_company")
df_produtoras  = spark.table("workspace.gold.dim_companies")

data_fim = date.today()
data_inicio = data_fim.replace(year=data_fim.year - ANOS)

# filmes lançados no período
df_filmes_periodo = (
    df_filmes
    .filter(F.col("status_filme") == "Lançado")
    .filter(F.col("data_lancamento").between(F.lit(data_inicio), F.lit(data_fim)))
    .select("sk_movie_id")
)

# lucro dos filmes do período, só com orçamento e receita informados
df_lucro_periodo = (
    df_filmes_periodo
    .join(df_fato, on="sk_movie_id")
    .filter(F.col("orcamento_usd").isNotNull() & F.col("receita_usd").isNotNull())
    .select("sk_movie_id", "lucro_usd", "lucro_brl")
)

#Debug de qntd de filmes
print("Filmes do período com dados financeiros:", df_lucro_periodo.count())

# liga cada filme com suas produtoras
df_lucro_produtora = (
    df_lucro_periodo
    .join(df_bridge, on="sk_movie_id")
    .join(df_produtoras, on="sk_company_id")
)

# lucro total por produtora
df_lucro_por_produtora = (
    df_lucro_produtora
    .groupBy("nome_produtora")
    .agg(
        F.sum("lucro_usd").alias("lucro_total_usd"),
        F.sum("lucro_brl").alias("lucro_total_brl"),
        F.countDistinct("sk_movie_id").alias("qtd_filmes"),
    )
)
display(df_lucro_por_produtora.orderBy(F.desc("lucro_total_usd")).limit(10))

Filmes do período com dados financeiros: 268


nome_produtora,lucro_total_usd,lucro_total_brl,qtd_filmes
Universal Pictures,5064624347.00,25738420931.44,20
Marvel Studios,4671219531.00,23739137656.54,7
Columbia Pictures,3157187163.00,16044825162.36,10
Illumination,2431353473.00,12356138349.79,3
Pascal Pictures,2305088862.00,11714461596.68,2
Kevin Feige Productions,2138205367.00,10866359675.09,4
Paramount,1935702405.00,9837239622.21,11
Lightstorm Entertainment,1860250281.00,9453791928.04,1
Skydance,1553532821.00,7895053796.32,2
Heyday Films,1490495872.00,7574700021.51,2
